<a href="https://colab.research.google.com/github/anubagar/anubagar/blob/main/Transformer_for_Translation_GREAT_LEARNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer for Translation - EN to HI


**Before you start:** `Runtime` -> `Change runtime type` -> `T4 GPU`.


## Install dependencies


In [ ]:
!pip install -q datasets nltk

#datasets - library containing many datasets, from that we import the 'iitb-english-hindi' dataset
#nltk - library used for evaluation purpose, here we use it to calculate 'BLEU' score

## Write the project files

### `show.py`


-This cell is for visualizing the outputs from different stages. It doesn't affect the transformer architecture.

In [ ]:
%%writefile show.py
#this command creates or overwrites a file named show.py

"""
Prints ONE example as it flows through every stage.

"""

import sys
from contextlib import contextmanager

import torch

ENABLED = False

_seen = set()
_name = ""
_kind = ""

WIDTH = 78


def init_console():
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass


def enable():
    global ENABLED
    ENABLED = True
    _seen.clear()


def disable():
    global ENABLED
    ENABLED = False


@contextmanager
def scope(name, kind=None):
    global _name, _kind
    prev_name, prev_kind = _name, _kind
    _name, _kind = name, (kind or name)
    try:
        yield
    finally:
        _name, _kind = prev_name, prev_kind


# --------------------------------------------------------------------------
# printing primitives
# --------------------------------------------------------------------------

def banner(title, char="="):
    print()
    print(char * WIDTH)
    print(title)
    print(char * WIDTH)


def note(text):
    print(f"    {text}")


def kv(label, value):
    print(f"    {label:<28} {value}")


def _corner(t, rows=3, cols=6):

    t = t.detach().float().cpu()

    if t.dim() == 4:        # (B, H, S, Dk) -> batch 0, head 0
        sub, where = t[0, 0, :rows, :cols], f"[0, 0, :{rows}, :{cols}]"
    elif t.dim() == 3:      # (B, S, D)     -> batch 0
        sub, where = t[0, :rows, :cols], f"[0, :{rows}, :{cols}]"
    elif t.dim() == 2:      # (S, D)
        sub, where = t[:rows, :cols], f"[:{rows}, :{cols}]"
    else:                   # (D,)
        sub, where = t[:cols], f"[:{cols}]"

    if sub.dim() == 1:
        sub = sub.unsqueeze(0)

    lines = ["  ".join(f"{v:9.4f}" for v in row) for row in sub]
    return where, lines


def step(label, tensor=None, rows=3, cols=6):
    if not ENABLED:
        return

    key = (_kind, label)
    if key in _seen:
        return
    _seen.add(key)

    title = f"{_name} | {label}" if _name else label

    if tensor is None:
        print(f"\n>> {title}")
        return

    shape = "x".join(str(s) for s in tensor.shape)
    print(f"\n>> {title}")
    print(f"   shape ({shape})   dtype {tensor.dtype}   device {tensor.device}")

    where, lines = _corner(tensor, rows, cols)
    print(f"   values{where}:")
    for line in lines:
        print(f"     {line}")


def matrix(label, m, rows=10, cols=10):
    if not ENABLED:
        return

    key = (_kind, label)
    if key in _seen:
        return
    _seen.add(key)

    m = m.detach().cpu()
    while m.dim() > 2:
        m = m[0]

    title = f"{_name} | {label}" if _name else label

    print(f"\n>> {title}")
    print(f"   showing top-left {min(rows, m.size(0))}x{min(cols, m.size(1))} "
          f"of ({m.size(0)}x{m.size(1)})")

    for row in m[:rows, :cols]:
        if m.dtype == torch.bool:
            print("     " + " ".join("1" if v else "." for v in row))
        else:
            print("     " + " ".join(f"{float(v):5.2f}" for v in row))


def tokens(label, toks, limit=20):
    if not ENABLED:
        return
    shown = list(toks)[:limit]
    tail = " ..." if len(toks) > limit else ""
    print(f"    {label:<28} {shown}{tail}")


Writing show.py


In [ ]:
%%writefile data.py


import json
import re
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader

SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


# --------------------------------------------------------------------------
# tokenization
# --------------------------------------------------------------------------

def tokenize(text, lang="en"):
    """
    English  : regex tokenization
    Hindi    : whitespace tokenization
    """
    text = text.lower()

    if lang == "hi":
        return text.strip().split()
    else:
        return re.findall(r"\w+|[^\w\s]", text)


# --------------------------------------------------------------------------
# vocabulary
# --------------------------------------------------------------------------

def build_vocab(sentences, lang, min_freq=2):
    counter = Counter() #specialized dictionary used for counting how many times each item occurs.

    for sent in sentences:
        counter.update(tokenize(sent, lang))



    vocab = {}
    idx = 0

    for tok in SPECIAL_TOKENS:
        vocab[tok] = idx
        idx += 1

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = idx
            idx += 1

    return vocab


def numericalize(text, vocab, lang):
    tokens = tokenize(text, lang)

    return (
        [vocab["<sos>"]] +
        [vocab.get(tok, vocab["<unk>"]) for tok in tokens] +
        [vocab["<eos>"]]
    )


def pad_sequence(seq, max_len, pad_idx):
    """
    Output:
        Tensor (max_len,)
    """
    seq = seq[:max_len]
    return #...


# --------------------------------------------------------------------------
# dataset / dataloader
# --------------------------------------------------------------------------

class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, src_vocab, tgt_vocab, max_len):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx): #given an sentence, it numericalize them and return output
        src_ids = numericalize(self.src_texts[idx], self.src_vocab, "en")
        tgt_ids = numericalize(self.tgt_texts[idx], self.tgt_vocab, "hi")

        src = pad_sequence(src_ids, self.max_len, self.src_vocab["<pad>"])
        tgt = pad_sequence(tgt_ids, self.max_len, self.tgt_vocab["<pad>"])

        return src, tgt


def collate_fn(batch):
    """
    Output:
        src: (B, S)
        tgt: (B, T)
    """
    src_batch, tgt_batch = zip(*batch)
    return torch.stack(src_batch), torch.stack(tgt_batch)


def make_loader(dataset, batch_size, shuffle, num_workers=0, pin_memory=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )


# --------------------------------------------------------------------------
# masks
# --------------------------------------------------------------------------

def make_src_mask(src, pad_idx):
    """
    src: (B, S)
    Output:
        src_mask: (B, 1, 1, S)
    """
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)


def make_tgt_mask(tgt, pad_idx):
    """
    tgt: (B, T)  [on GPU or CPU]

    Output:
        tgt_mask: (B, 1, T, T)
    """
    B, T = tgt.shape
    device = tgt.device   # key line

    pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
    # pad_mask: (B, 1, 1, T)

    causal_mask = torch.tril(
        torch.ones(T, T, device=device)
    ).bool()
    # causal_mask: (T, T)

    return pad_mask & causal_mask


# --------------------------------------------------------------------------
# local file IO (replaces the Colab `load_dataset` call at training time)
# --------------------------------------------------------------------------

def load_pairs(path):

    src, tgt = [], []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            src.append(row["en"])
            tgt.append(row["hi"])

    if not src:
        raise RuntimeError(f"{path} is empty - run prepare_data.py first")

    return src, tgt


Writing data.py


### `model.py`


**### Sinusoidal Positional Encoding (Mathematical Form)**

Given a sequence position $ p \in \{0, 1, \dots, L-1\} $ and model dimension $ d_{\text{model}} $, the positional encoding vector
$ \mathrm{PE}(p) \in \mathbb{R}^{d_{\text{model}}} $ is defined as:

$$
\mathrm{PE}(p, 2i) = \sin\!\left(p / 10000^{-\frac{2i}{d_{\text{model}}}}\right)
$$

$$
\mathrm{PE}(p, 2i+1) = \cos\!\left(p / 10000^{-\frac{2i}{d_{\text{model}}}}\right)
$$

for $ i = 0, 1, \dots, \left\lfloor \frac{d_{\text{model}}}{2} \right\rfloor - 1 $.

The input embedding $ x_p \in \mathbb{R}^{d_{\text{model}}} $ at position $ p $ is augmented as:

$$
\tilde{x}_p = x_p + \mathrm{PE}(p)
$$

d_model = 512

max_length = 64

### Why use `log` and `exp`?

The positional encoding requires the term

$$
10000^{-\frac{2i}{d_{\text{model}}}}
$$

Using the identity

$$
a^b = e^{b\ln(a)}
$$

we can rewrite it as

$$
\boxed{
10000^{-\frac{2i}{d_{\text{model}}}}
=
\exp\left(
-\frac{2i}{d_{\text{model}}}\log(10000)
\right)
}
$$

This allows us to compute all the scaling/frequency factors efficiently using tensor operations:


In [ ]:
%%writefile model.py


import math

import torch
import torch.nn as nn
import torch.nn.functional as F

import show


class PositionalEncoding(nn.Module):
  #start with matching the formula and the trick of making it happen
    def __init__(self, d_model, max_len):
        super().__init__() #it initiates the nn.module

        pe = torch.zeros(max_len, d_model) #it creates a matrix of size max_len, d_model as a sentence will be represented as matrix of size '64x512'
        position = torch.arange(0, max_len).unsqueeze(1)  #creates a tensor of size (max_len(64), 1). Later used for mentioning the position of the words/tokens in the sequence.
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model) #this exponential and log conversion is for algebraic advantage.
        ) #this will output a vector of dimension d_model/2, this is that 10000 power term. Here instead of using 2i and changing i from 0,1,2.., we changing it with 0,2,4..

        pe[:, 0::2] = torch.sin(position * div_term)#broadcasting is used here, calculates the sin position and add it alternatively columns starting from 0,2,4
        pe[:, 1::2] = torch.cos(position * div_term)##calculates the cos posit1ion and add it alternatively columns starting from 1,3,5

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)  #keeps the model and embedding in same device, also mention that the values of the matrix are fixed, it doesn't need gradient update

    def forward(self, x):
        """
        x: (B, seq_len, D)
        """
        seq_len = x.size(1) #dimension 1, represents the number of tokens

        if seq_len > self.pe.size(1):
            raise ValueError(
                f"sequence length {seq_len} exceeds max_len {self.pe.size(1)} "
                f"that the positional encoding table was built with"
            )

        show.step("token embeddings (before positions)", x)
        show.step("sinusoidal table pe[:, :seq_len, :]", self.pe[:, :seq_len, :])

        out = x + self.pe[:, :seq_len, :] #same  positional encoding is added to  each sequence in a batch, positional encoding is added only to the vectors corresponding to the tokens not the paddings

        show.step("embeddings + positional encoding", out)
        return out


def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q: (B, H, S, Dk) #means for each of 32 sentence in a batch, we have 'H' different Q,K,V matrix of size (S,Dk)
    # K: (B, H, S, Dk)
    # V: (B, H, S, Dk)

    d_k = Q.size(-1) #extracts dimension

    scores = torch.matmul(Q, K.transpose(-2, -1)) # K.transpose(-2, -1), swaps last two dimension
    # K^T: (B, H, Dk, S)
    # scores: (B, H, S, S)

    show.step("raw scores  Q @ K^T", scores)

    scores = scores / math.sqrt(d_k)
    # scores: (B, H, S, S)

    show.step(f"scaled scores  / sqrt(d_k)={math.sqrt(d_k):.2f}", scores)
    #the masking is done to ensure that the <PAD> token is not involved for softmax
    if mask is not None: #if that entry have None = True, not None = False(implies, value = 0)
        # -1e9 for float32

        fill = -1e9 if scores.dtype == torch.float32 else torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask == 0, fill)
        # mask: broadcastable to (B, H, S, S)

        show.step("scores after masking", scores)

    attn_weights = F.softmax(scores, dim=-1) #says that softmax to be applied row-wise.
    # attn_weights: (B, H, S, S)

    show.step("attention weights = softmax(scores)", attn_weights)
    show.matrix("attention weights, head 0 (each row sums to 1)", attn_weights[0, 0])

    output = torch.matmul(attn_weights, V)
    # output: (B, H, S, Dk)

    show.step("attention output  weights @ V", output)

    return output, attn_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, name="attention", kind=None):
        super().__init__()

        assert d_model % num_heads == 0 #ensures that the constraint of proper splitting of heads got satisfied

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.name = name
        self.kind = kind or name

        #these are linear tarnsformation that gives the Q, K, V matrices. This weight matrix is weights of fully connected neural network.
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        #the final transformation after concatenation that combines the embedding from the all the heads
        self.W_o = nn.Linear(d_model, d_model)


    #this takes the input 'X' and splits the input into 'H' parts(i.e the dimension 512 will splitted into 'H' parts.)
    def split_heads(self, x):
        # x: (B, S, D)
        B, S, _ = x.size() #it will return B,S,D. The dimesion D is not needed ", _" will ignore the last value

        x = x.view(B, S, self.num_heads, self.d_k)
        # x: (B, S, H, Dk), in this shape for a given 'X' H split is maded.

        x = x.transpose(1, 2)
         # x: (B, H, S, Dk), after transpose, the H splits of each token is distributed among H heads with each head carrying one split to it

        return x

    def forward(self, Q, K, V, mask=None): #here Q = K = V = X i.e. the input matrix
        # Q, K, V: (B, S, D)

        with show.scope(self.name, self.kind):
          #creating a one master Q,K,V matrix
            Q = self.W_q(Q)
            # Q: (B, S, D)

            K = self.W_k(K)
            # K: (B, S, D)

            V = self.W_v(V)
            # V: (B, S, D)

            show.step("Q = x @ W_q", Q)
            show.step("K = x @ W_k", K)
            show.step("V = x @ W_v", V)


            #splitting the Q,K,V among H heads
            Q = self.split_heads(Q)
            # Q: (B, H, S, Dk)

            K = self.split_heads(K)
            # K: (B, H, S, Dk)

            V = self.split_heads(V)
            # V: (B, H, S, Dk)

            show.step(f"Q split into {self.num_heads} heads of d_k={self.d_k}", Q)

            attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)
            # attn_output: (B, H, S, Dk)

            attn_output = attn_output.transpose(1, 2)
            # attn_output: (B, S, H, Dk)

            attn_output = attn_output.contiguous().view(
                Q.size(0), -1, self.d_model
            )#view() - will reshape the tensor, the dimension in the position -1 will calculated automatically.
            #in this we are explicitly mentioning the first and last dimension, the -1 in middle place automatically calculates middle dimension
            # .contiguous() is related to memory which ensures that it is in required format b4 transpose
            # attn_output: (B, S, D)

            show.step("heads concatenated back to d_model", attn_output)

            output = self.W_o(attn_output)
            # output: (B, S, D)

            show.step("output projection  @ W_o", output)

            return output


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        #d_ff id hidder layer dimension
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # x: (B, S, D)

        x = self.fc1(x)
        # x: (B, S, d_ff)

        show.step("FFN: expand  fc1(x)", x)

        x = F.relu(x)
        # x: (B, S, d_ff)

        show.step("FFN: relu", x)

        x = self.fc2(x)
        # x: (B, S, D)

        show.step("FFN: project back  fc2(x)", x)

        return x


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, layer_idx=0):
        super().__init__()

        self.name = f"Encoder layer {layer_idx + 1}"

        self.self_attn = MultiHeadAttention(
            d_model, num_heads,
            name=f"{self.name} | self-attention", kind="encoder.self_attn",
        )
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (B, S, D)

        attn = self.self_attn(x, x, x, mask)
        # attn: (B, S, D)

        with show.scope(self.name, "encoder.layer"):
            x = x + self.dropout(attn)
            # x: (B, S, D)

            show.step("residual  x + dropout(attn)", x)

            x = self.norm1(x)
            # x: (B, S, D)

            show.step("layer norm 1", x)

            ffn = self.ffn(x)
            # ffn: (B, S, D)

            x = x + self.dropout(ffn)
            # x: (B, S, D)

            show.step("residual  x + dropout(ffn)", x)

            x = self.norm2(x)
            # x: (B, S, D)

            show.step("layer norm 2 -> encoder layer output", x)

            return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, layer_idx=0):
        super().__init__()

        self.name = f"Decoder layer {layer_idx + 1}"

        self.self_attn = MultiHeadAttention(
            d_model, num_heads,
            name=f"{self.name} | masked self-attention", kind="decoder.self_attn",
        )
        self.cross_attn = MultiHeadAttention(
            d_model, num_heads,
            name=f"{self.name} | cross-attention", kind="decoder.cross_attn",
        )
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        # x: (B, T, D)
        # enc_out: (B, S, D)

        attn1 = self.self_attn(x, x, x, tgt_mask)
        # attn1: (B, T, D)

        with show.scope(self.name, "decoder.layer"):
            x = self.norm1(x + self.dropout(attn1))
            # x: (B, T, D)

            show.step("after masked self-attn + residual + norm", x)

        attn2 = self.cross_attn(x, enc_out, enc_out, src_mask)
        # attn2: (B, T, D)

        with show.scope(self.name, "decoder.layer"):
            x = self.norm2(x + self.dropout(attn2))
            # x: (B, T, D)

            show.step("after cross-attn + residual + norm", x)

            ffn = self.ffn(x)
            # ffn: (B, T, D)

            x = self.norm3(x + self.dropout(ffn))
            # x: (B, T, D)

            show.step("after FFN + residual + norm -> decoder layer output", x)

            return x


class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()
        #the innput is (B - batch size, S - sequence length) the output is (B, S, D - dimension of embedding). Converts word into embedding vector

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        #multiple encoder layer is made using List comprehension
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout, layer_idx=i)
            for i in range(num_layers)
        ])

    def forward(self, x, mask=None):
        """
        x: (B, S)
        """
        with show.scope("Encoder", "encoder"):
            show.step("source token ids", x)

            x = self.embedding(x)          # (B, S, D)
            x = self.pos_encoding(x)       # (B, S, D)

        for layer in self.layers:
            x = layer(x, mask)             # (B, S, D)

        with show.scope("Encoder", "encoder"):
            show.step("encoder output (the memory the decoder attends to)", x)

        return x


class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout, layer_idx=i)
            for i in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        """
        x: (B, T)
        enc_out: (B, S, D)
        """
        with show.scope("Decoder", "decoder"):
            show.step("target token ids (shifted right)", x)

            x = self.embedding(x)          # (B, T, D)
            x = self.pos_encoding(x)       # (B, T, D)

        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
            # (B, T, D)

        with show.scope("Decoder", "decoder"):
            show.step("decoder output", x)

            logits = self.fc_out(x)        # (B, T, vocab_size)

            show.step("logits over the target vocabulary", logits)

        return logits


class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        max_len,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        dropout=0.1,
    ):
        super().__init__()

        self.encoder = Encoder(
            src_vocab_size,
            max_len,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout,
        )

        self.decoder = Decoder(
            tgt_vocab_size,
            max_len,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout,
        )

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        src: (B, S)
        tgt: (B, T)

        src_mask: (B, 1, 1, S)
        tgt_mask: (B, 1, T, T)
        """

        enc_out = self.encoder(src, src_mask)
        # enc_out: (B, S, D)

        out = self.decoder(tgt, enc_out, src_mask, tgt_mask)
        # out: (B, T, tgt_vocab_size)

        return out


Writing model.py


### `decoding.py`

Greedy, sampling (temperature / top-k / top-p) and beam search decoding.

In [ ]:
%%writefile decoding.py


import torch
import torch.nn.functional as F

from data import make_src_mask, make_tgt_mask, numericalize, pad_sequence


def _ids_to_text(tgt_ids, tgt_vocab):
    idx2word_tgt = {idx: tok for tok, idx in tgt_vocab.items()}

    tokens = [
        idx2word_tgt[idx]
        for idx in tgt_ids
        if idx not in (
            tgt_vocab["<sos>"],
            tgt_vocab["<eos>"],
            tgt_vocab["<pad>"],
        )
    ]

    return " ".join(tokens)


def _encode_source(src_sentence, src_vocab, max_len, device):
    src_ids = numericalize(src_sentence, src_vocab, lang="en")
    src = pad_sequence(src_ids, max_len, src_vocab["<pad>"])
    src = src.unsqueeze(0).to(device)
    # src: (1, S)

    src_mask = make_src_mask(src, src_vocab["<pad>"])
    return src, src_mask


def greedy_decode(
    model,
    src_sentence,
    src_vocab,
    tgt_vocab,
    max_len,
    device,
):
    """
    src_sentence: string (English)

    Returns:
        decoded_sentence: string (Hindi)
    """

    model.eval()

    # ---- Encode source sentence ----
    src, src_mask = _encode_source(src_sentence, src_vocab, max_len, device)

    # ---- Start decoding ----
    tgt_ids = [tgt_vocab["<sos>"]]

    for t in range(max_len - 1):
        tgt = torch.tensor(tgt_ids).unsqueeze(0).to(device)
        # tgt: (1, t+1)

        tgt_mask = make_tgt_mask(tgt, tgt_vocab["<pad>"])

        with torch.no_grad():
            logits = model(src, tgt, src_mask, tgt_mask)
            # logits: (1, t+1, vocab_size)

        next_token_logits = logits[0, -1]      # (vocab_size,)
        next_token = next_token_logits.argmax().item()

        tgt_ids.append(next_token)

        if next_token == tgt_vocab["<eos>"]:
            break

    # ---- Convert ids to tokens ----
    return _ids_to_text(tgt_ids, tgt_vocab)


def sample_from_logits(
    logits,
    temperature=1.0,
    top_k=None,
    top_p=None
):
    """
    Samples a token from logits using optional temperature, top-k, or top-p.

    Args:
        logits: Tensor of shape (vocab_size,)
        temperature: float (>0). Lower = more deterministic
        top_k: int or None. Keep only top-k tokens
        top_p: float or None. Nucleus sampling threshold

    Returns:
        sampled_token_id: int
    """

    # ---- Temperature scaling ----
    logits = logits / temperature

    # ---- Convert to probabilities ----
    probs = F.softmax(logits, dim=-1)

    # ---- Top-k filtering ----
    if top_k is not None:
        top_k = min(top_k, probs.size(-1))
        values, indices = torch.topk(probs, top_k)
        probs_filtered = torch.zeros_like(probs)
        probs_filtered[indices] = values
        probs = probs_filtered / probs_filtered.sum()

    # ---- Top-p (nucleus) filtering ----
    if top_p is not None:
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # Keep tokens until cumulative prob exceeds p
        cutoff = cumulative_probs > top_p
        cutoff[..., 1:] = cutoff[..., :-1].clone()
        cutoff[..., 0] = False

        sorted_probs[cutoff] = 0.0
        probs = torch.zeros_like(probs)
        probs[sorted_indices] = sorted_probs
        probs = probs / probs.sum()

    # ---- Sample ----
    return torch.multinomial(probs, num_samples=1).item()


def sampling_decode(
    model,
    src_sentence,
    src_vocab,
    tgt_vocab,
    max_len,
    device,
    temperature=1.0,
    top_k=None,
    top_p=None
):
    """
    Sampling-based decoding.

    Supports:
        - Temperature sampling
        - Top-k sampling
        - Nucleus (top-p) sampling
    """

    model.eval()

    src, src_mask = _encode_source(src_sentence, src_vocab, max_len, device)

    tgt_ids = [tgt_vocab["<sos>"]]

    for _ in range(max_len - 1):
        tgt = torch.tensor(tgt_ids).unsqueeze(0).to(device)
        tgt_mask = make_tgt_mask(tgt, tgt_vocab["<pad>"])

        with torch.no_grad():
            logits = model(src, tgt, src_mask, tgt_mask)

        next_logits = logits[0, -1]

        next_token = sample_from_logits(
            next_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

        tgt_ids.append(next_token)

        if next_token == tgt_vocab["<eos>"]:
            break

    return _ids_to_text(tgt_ids, tgt_vocab)


def beam_search_decode(
    model,
    src_sentence,
    src_vocab,
    tgt_vocab,
    max_len,
    device,
    beam_size=5
):
    """
    Beam search decoding.

    Keeps top-K hypotheses based on log-probability.
    """

    model.eval()

    src, src_mask = _encode_source(src_sentence, src_vocab, max_len, device)

    # Each beam: (token_ids, log_prob)
    beams = [([tgt_vocab["<sos>"]], 0.0)]

    for _ in range(max_len - 1):
        new_beams = []

        for tokens, score in beams:
            if tokens[-1] == tgt_vocab["<eos>"]:
                new_beams.append((tokens, score))
                continue

            tgt = torch.tensor(tokens).unsqueeze(0).to(device)
            tgt_mask = make_tgt_mask(tgt, tgt_vocab["<pad>"])

            with torch.no_grad():
                logits = model(src, tgt, src_mask, tgt_mask)

            log_probs = F.log_softmax(logits[0, -1], dim=-1)

            topk_log_probs, topk_ids = torch.topk(log_probs, beam_size)

            for log_p, idx in zip(topk_log_probs, topk_ids):
                new_tokens = tokens + [idx.item()]
                new_score = score + log_p.item()
                new_beams.append((new_tokens, new_score))

        # Keep best beams
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        # Early stop if all beams ended
        if all(tokens[-1] == tgt_vocab["<eos>"] for tokens, _ in beams):
            break

    best_tokens = beams[0][0]

    return _ids_to_text(best_tokens, tgt_vocab)


def translate(model, sentence, src_vocab, tgt_vocab, max_len, device,
              method="greedy", temperature=1.0, top_k=None, top_p=None,
              beam_size=5):
    """Single entry point used by train.py and translate.py."""
    if method == "greedy":
        return greedy_decode(model, sentence, src_vocab, tgt_vocab, max_len, device)

    if method == "sample":
        return sampling_decode(
            model, sentence, src_vocab, tgt_vocab, max_len, device,
            temperature=temperature, top_k=top_k, top_p=top_p,
        )

    if method == "beam":
        return beam_search_decode(
            model, sentence, src_vocab, tgt_vocab, max_len, device,
            beam_size=beam_size,
        )

    raise ValueError(f"unknown decoding method: {method}")


Writing decoding.py


### `checkpoint.py`


In [ ]:
%%writefile checkpoint.py


import os

import torch

from model import Transformer


def save_checkpoint(path, model, vocab_src, vocab_tgt, config,
                    optimizer=None, epoch=None, loss=None):
    directory = os.path.dirname(os.path.abspath(path))
    os.makedirs(directory, exist_ok=True)

    payload = {
        "model_state_dict": model.state_dict(),
        "vocab_src": vocab_src,
        "vocab_tgt": vocab_tgt,
        "config": config,
        "epoch": epoch,
        "loss": loss,
    }

    if optimizer is not None:
        payload["optimizer_state_dict"] = optimizer.state_dict()

    torch.save(payload, path)
    return path


def load_checkpoint(path, device, with_optimizer=False):
    """Rebuild the model exactly as it was trained and load the weights."""
    ckpt = torch.load(path, map_location=device, weights_only=False)

    config = ckpt["config"]
    vocab_src = ckpt["vocab_src"]
    vocab_tgt = ckpt["vocab_tgt"]

    model = Transformer(
        src_vocab_size=len(vocab_src),
        tgt_vocab_size=len(vocab_tgt),
        max_len=config["max_len"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        d_ff=config["d_ff"],
        dropout=config.get("dropout", 0.1),
    ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    if with_optimizer:
        return model, vocab_src, vocab_tgt, config, ckpt

    return model, vocab_src, vocab_tgt, config


Writing checkpoint.py


### `walkthrough.py`

The stage-by-stage walkthrough of one example.

In [ ]:
%%writefile walkthrough.py


import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F

#import gpu
import show
from data import (
    SPECIAL_TOKENS,
    TranslationDataset,
    build_vocab,
    collate_fn,
    load_pairs,
    make_src_mask,
    make_tgt_mask,
    numericalize,
    pad_sequence,
    tokenize,
)
from model import Transformer


def run_walkthrough(model, train_en, train_hi, vocab_src, vocab_tgt,
                    max_len, device, criterion=None, index=0):
    """Push one example through every stage and print what comes out."""

    idx2word_tgt = {i: t for t, i in vocab_tgt.items()}
    was_training = model.training
    model.eval()

    # ---------------------------------------------------------------- 1
    show.banner("STAGE 1 | RAW DATASET EXAMPLE")
    src_text = train_en[index]
    tgt_text = train_hi[index]
    show.kv("english (source)", src_text)
    show.kv("hindi (target)", tgt_text)
    show.kv("pairs available", len(train_en))

    # ---------------------------------------------------------------- 2
    show.banner("STAGE 2 | TOKENIZER")
    src_tokens = tokenize(src_text, "en")
    tgt_tokens = tokenize(tgt_text, "hi")
    show.kv("en: regex \\w+|[^\\w\\s]", src_tokens)
    show.kv("hi: whitespace split", tgt_tokens)
    show.kv("token counts", f"en={len(src_tokens)}  hi={len(tgt_tokens)}")

    # ---------------------------------------------------------------- 3
    show.banner("STAGE 3 | VOCABULARY")
    show.kv("english vocab size", len(vocab_src))
    show.kv("hindi vocab size", len(vocab_tgt))
    show.kv("special tokens", {t: vocab_src[t] for t in SPECIAL_TOKENS})
    first_words = list(vocab_src)[4:12]
    show.kv("first real en words", {w: vocab_src[w] for w in first_words})
    unknown = [t for t in src_tokens if t not in vocab_src]
    show.kv("tokens mapped to <unk>", unknown if unknown else "none")

    # ---------------------------------------------------------------- 4
    show.banner("STAGE 4 | NUMERICALIZE  (<sos> ... <eos>)")
    src_ids = numericalize(src_text, vocab_src, "en")
    tgt_ids = numericalize(tgt_text, vocab_tgt, "hi")
    show.kv("source ids", src_ids)
    show.kv("target ids", tgt_ids)
    show.kv("target ids -> tokens", [idx2word_tgt[i] for i in tgt_ids])

    # ---------------------------------------------------------------- 5
    show.banner(f"STAGE 5 | PADDING TO max_len={max_len}")
    src_padded = pad_sequence(src_ids, max_len, vocab_src["<pad>"])
    tgt_padded = pad_sequence(tgt_ids, max_len, vocab_tgt["<pad>"])
    show.kv("padded source", src_padded.tolist())
    show.kv("real vs pad", f"{len(src_ids)} real + "
                           f"{max_len - len(src_ids)} <pad>")

    # ---------------------------------------------------------------- 6
    show.banner("STAGE 6 | BATCHING  (collate_fn)")
    dataset = TranslationDataset(train_en, train_hi, vocab_src, vocab_tgt, max_len)
    batch = collate_fn([dataset[index], dataset[index + 1]])
    src, tgt = batch[0].to(device), batch[1].to(device)
    show.kv("src batch shape", tuple(src.shape))
    show.kv("tgt batch shape", tuple(tgt.shape))

    # ---------------------------------------------------------------- 7
    show.banner("STAGE 7 | TEACHER FORCING SPLIT")
    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]
    show.kv("decoder input  tgt[:, :-1]",
            [idx2word_tgt[i] for i in tgt_input[0, :8].tolist()])
    show.kv("expected label tgt[:, 1:]",
            [idx2word_tgt[i] for i in tgt_output[0, :8].tolist()])
    show.note("the decoder sees position t and must predict position t+1")

    # ---------------------------------------------------------------- 8
    show.banner("STAGE 8 | MASKS")
    src_mask = make_src_mask(src, vocab_src["<pad>"])
    tgt_mask = make_tgt_mask(tgt_input, vocab_tgt["<pad>"])
    show.kv("src_mask shape", tuple(src_mask.shape))
    show.kv("tgt_mask shape", tuple(tgt_mask.shape))

    show.enable()
    show.matrix("src_mask: 1 = real token, . = <pad>",
                src_mask[0, 0], rows=1, cols=24)
    show.matrix("tgt_mask: causal + padding (row t sees only <= t)",
                tgt_mask[0, 0], rows=10, cols=10)

    # ---------------------------------------------------------------- 9
    show.banner("STAGE 9 | FORWARD PASS THROUGH THE MODEL")
    n_layers = len(model.encoder.layers)
    show.note(f"repeated blocks are printed once - layer 1 stands in for all {n_layers}")

    with torch.no_grad():
        logits = model(src, tgt_input, src_mask, tgt_mask)

    show.disable()

    # --------------------------------------------------------------- 10
    show.banner("STAGE 10 | LOGITS -> PREDICTED TOKEN")
    probs = F.softmax(logits[0, 0], dim=-1)
    top_p, top_i = torch.topk(probs, 5)
    show.kv("predicting position", "1 (first real target token)")
    show.kv("gold token", idx2word_tgt[tgt_output[0, 0].item()])
    show.note("top-5 candidates:")
    for p, i in zip(top_p.tolist(), top_i.tolist()):
        print(f"        {idx2word_tgt[i]:<20} p={p:.4f}")

    greedy_ids = logits[0].argmax(dim=-1)[:12].tolist()
    show.kv("greedy tokens (teacher forced)",
            [idx2word_tgt[i] for i in greedy_ids])

    # --------------------------------------------------------------- 11
    if criterion is not None:
        show.banner("STAGE 11 | LOSS")
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1),
        )
        show.kv("cross entropy (pad ignored)", f"{loss.item():.4f}")
        show.kv("random-guess baseline", f"{torch.log(torch.tensor(float(len(vocab_tgt)))).item():.4f}")

    show.banner("END OF WALKTHROUGH")

    if was_training:
        model.train()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--checkpoint", default=None,
                        help="load trained weights instead of a random model")
    parser.add_argument("--index", type=int, default=0)
    parser.add_argument("--max-len", type=int, default=64)
    parser.add_argument("--min-freq", type=int, default=2)
    parser.add_argument("--d-model", type=int, default=512)
    parser.add_argument("--num-layers", type=int, default=6)
    parser.add_argument("--num-heads", type=int, default=8)
    parser.add_argument("--d-ff", type=int, default=2048)
    parser.add_argument("--device", default="auto",
                        help="auto (freest GPU) / cuda:2 / cpu")
    args = parser.parse_args()

    show.init_console()

    device = gpu.resolve_device(args.device)

    train_en, train_hi = load_pairs(f"{args.data_dir}/train.jsonl")

    if args.checkpoint:
        from checkpoint import load_checkpoint
        model, vocab_src, vocab_tgt, config = load_checkpoint(args.checkpoint, device)
        max_len = config["max_len"]
        show.kv("loaded checkpoint", args.checkpoint)
    else:
        vocab_src = build_vocab(train_en, "en", args.min_freq)
        vocab_tgt = build_vocab(train_hi, "hi", args.min_freq)
        max_len = args.max_len
        model = Transformer(
            src_vocab_size=len(vocab_src),
            tgt_vocab_size=len(vocab_tgt),
            max_len=max_len,
            d_model=args.d_model,
            num_layers=args.num_layers,
            num_heads=args.num_heads,
            d_ff=args.d_ff,
        ).to(device)
        show.note("using an UNTRAINED model - the numbers are random, "
                  "the shapes are what matter")

    criterion = nn.CrossEntropyLoss(ignore_index=vocab_tgt["<pad>"])

    run_walkthrough(model, train_en, train_hi, vocab_src, vocab_tgt,
                    max_len, device, criterion, index=args.index)


if __name__ == "__main__":
    main()


Writing walkthrough.py


### `prepare_data.py`

Downloads the EN-HI pairs once and caches them as JSONL.

In [ ]:
%%writefile prepare_data.py


import argparse
import json
import os
from itertools import islice

import show


def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for en, hi in rows:
            f.write(json.dumps({"en": en, "hi": hi}, ensure_ascii=False) + "\n")


def extract_pairs(dataset):
    src, tgt = [], []
    for sample in dataset:
        src.append(sample["translation"]["en"])
        tgt.append(sample["translation"]["hi"])
    return src, tgt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", default="cfilt/iitb-english-hindi")
    parser.add_argument("--train-size", type=int, default=10000)
    parser.add_argument("--test-size", type=int, default=100)
    parser.add_argument("--data-dir", default="data")
    parser.add_argument(
        "--full-download",
        action="store_true",
        help="download the whole corpus instead of streaming the first rows",
    )
    args = parser.parse_args()

    show.init_console()

    from datasets import load_dataset

    total = args.train_size + args.test_size

    show.banner(f"DOWNLOADING {total} PAIRS FROM {args.dataset}")

    if args.full_download:
        dataset_full = load_dataset(args.dataset)
        dataset_small = dataset_full["train"].select(range(total))

        train_raw = dataset_small.select(range(args.train_size))
        test_raw = dataset_small.select(range(args.train_size, total))

        train_en, train_hi = extract_pairs(train_raw)
        test_en, test_hi = extract_pairs(test_raw)
    else:
        stream = load_dataset(args.dataset, split="train", streaming=True)
        rows = list(islice(stream, total))

        en, hi = extract_pairs(rows)
        train_en, train_hi = en[:args.train_size], hi[:args.train_size]
        test_en, test_hi = en[args.train_size:], hi[args.train_size:]

    train_path = os.path.join(args.data_dir, "train.jsonl")
    test_path = os.path.join(args.data_dir, "test.jsonl")

    write_jsonl(train_path, zip(train_en, train_hi))
    write_jsonl(test_path, zip(test_en, test_hi))

    show.kv("train pairs", f"{len(train_en)}  ->  {train_path}")
    show.kv("test pairs", f"{len(test_en)}  ->  {test_path}")

    show.banner("ONE EXAMPLE FROM THE DOWNLOADED DATA", "-")
    show.kv("EN", train_en[5])
    show.kv("HI", train_hi[5])


if __name__ == "__main__":
    main()


Writing prepare_data.py


### `train.py`


 tgt_input = [y0, y1, y2,..........yT-1]

 tgt_output = [y1, y2,.........yT]

In [ ]:
%%writefile train.py


import argparse
import os
import sys
import time

import torch
import torch.nn as nn

import gpu
import show
from checkpoint import load_checkpoint, save_checkpoint
from data import (
    TranslationDataset,
    build_vocab,
    load_pairs,
    make_loader,
    make_src_mask,
    make_tgt_mask,
)
from decoding import greedy_decode
from model import Transformer
from walkthrough import run_walkthrough


def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--data-dir", default="data")
    p.add_argument("--out", default="checkpoints/transformer_en_hi.pt")

    # architecture - the notebook's defaults
    p.add_argument("--max-len", type=int, default=64)
    p.add_argument("--d-model", type=int, default=512)
    p.add_argument("--num-layers", type=int, default=6)
    p.add_argument("--num-heads", type=int, default=8)
    p.add_argument("--d-ff", type=int, default=2048)
    p.add_argument("--dropout", type=float, default=0.1)
    p.add_argument("--min-freq", type=int, default=2)

    # optimisation
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--lr", type=float, default=1e-4)

    # runtime
    p.add_argument("--device", default="auto",
                   help="auto (freest GPU) / cuda:2 / cpu")
    p.add_argument("--list-gpus", action="store_true",
                   help="print the visible GPUs and exit")
    p.add_argument("--num-workers", type=int, default=0,
                   help="DataLoader workers (2-4 helps on a Linux server)")
    p.add_argument("--amp", action="store_true",
                   help="mixed precision - less VRAM, faster on RTX cards")
    p.add_argument("--log-every", type=int, default=20)
    p.add_argument("--resume", default=None, help="checkpoint to continue from")
    p.add_argument("--no-walkthrough", action="store_true")
    p.add_argument("--sample-sentence", default="an empty slot on the tableau")

    return p.parse_args()


@torch.no_grad()
def evaluate_loss(model, loader, criterion, device, pad_src, pad_tgt):
    model.eval()
    total = 0.0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_mask = make_src_mask(src, pad_src)
        tgt_mask = make_tgt_mask(tgt_input, pad_tgt)

        logits = model(src, tgt_input, src_mask, tgt_mask)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1),
        )
        total += loss.item()

    model.train()
    return total / max(len(loader), 1)


def main():
    args = parse_args()
    show.init_console()

    if args.list_gpus:
        gpu.describe_devices()
        return

    device = gpu.resolve_device(args.device)
    gpu.describe_selected(device)

    use_amp = args.amp and device.type == "cuda"

    # ---------------------------------------------------------------- data
    train_path = os.path.join(args.data_dir, "train.jsonl")
    test_path = os.path.join(args.data_dir, "test.jsonl")

    if not os.path.exists(train_path):
        print(f"\n{train_path} not found. Run:  python prepare_data.py")
        sys.exit(1)

    train_en, train_hi = load_pairs(train_path)
    test_en, test_hi = load_pairs(test_path)

    # ------------------------------------------------------- model + vocab
    if args.resume:
        model, vocab_src, vocab_tgt, config, ckpt = load_checkpoint(
            args.resume, device, with_optimizer=True
        )
        start_epoch = (ckpt.get("epoch") or 0)
        max_len = config["max_len"]
        show.kv("resumed from", f"{args.resume} (epoch {start_epoch})")
    else:
        vocab_src = build_vocab(train_en, lang="en", min_freq=args.min_freq)
        vocab_tgt = build_vocab(train_hi, lang="hi", min_freq=args.min_freq)
        max_len = args.max_len
        start_epoch = 0
        ckpt = None

        config = {
            "max_len": max_len,
            "d_model": args.d_model,
            "num_layers": args.num_layers,
            "num_heads": args.num_heads,
            "d_ff": args.d_ff,
            "dropout": args.dropout,
            "min_freq": args.min_freq,
        }

        model = Transformer(
            src_vocab_size=len(vocab_src),
            tgt_vocab_size=len(vocab_tgt),
            d_model=args.d_model,
            num_heads=args.num_heads,
            num_layers=args.num_layers,
            d_ff=args.d_ff,
            max_len=max_len,
            dropout=args.dropout,
        ).to(device)

    PAD_SRC = vocab_src["<pad>"]
    PAD_TGT = vocab_tgt["<pad>"]

    train_dataset = TranslationDataset(train_en, train_hi, vocab_src, vocab_tgt, max_len)
    test_dataset = TranslationDataset(test_en, test_hi, vocab_src, vocab_tgt, max_len)

    pin = device.type == "cuda"
    train_loader = make_loader(train_dataset, args.batch_size, shuffle=True,
                               num_workers=args.num_workers, pin_memory=pin)
    test_loader = make_loader(test_dataset, args.batch_size, shuffle=False,
                              num_workers=args.num_workers, pin_memory=pin)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TGT)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    if ckpt is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    n_params = sum(p.numel() for p in model.parameters())

    show.banner("SETUP")
    show.kv("english vocab", len(vocab_src))
    show.kv("hindi vocab", len(vocab_tgt))
    show.kv("train / test pairs", f"{len(train_dataset)} / {len(test_dataset)}")
    show.kv("batches per epoch", len(train_loader))
    show.kv("parameters", f"{n_params:,}  ({n_params * 4 / 1024**2:.0f} MB fp32)")
    show.kv("mixed precision", use_amp)

    # -------------------------------------------------- stage walkthrough
    if not args.no_walkthrough:
        run_walkthrough(model, train_en, train_hi, vocab_src, vocab_tgt,
                        max_len, device, criterion)

    # ------------------------------------------------------------ training
    show.banner("TRAINING")

    best_loss = float("inf")
    best_path = args.out.replace(".pt", "_best.pt")

    for epoch in range(start_epoch, start_epoch + args.epochs):
        model.train()
        total_loss = 0.0
        started = time.time()

        for step, (src, tgt) in enumerate(train_loader, start=1):
            """
            src: (B, S)
            tgt: (B, T)
            """
            src = src.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)

            #teacher enforcing

            tgt_input = tgt[:, :-1]      # (B, T-1)
            tgt_output = tgt[:, 1:]      # (B, T-1)

            src_mask = make_src_mask(src, PAD_SRC)
            tgt_mask = make_tgt_mask(tgt_input, PAD_TGT)

            with torch.amp.autocast(device.type, enabled=use_amp):
                logits = model(src, tgt_input, src_mask, tgt_mask)
                # logits: (B, T-1, vocab_size)

                loss = criterion(
                    logits.reshape(-1, logits.size(-1)),
                    tgt_output.reshape(-1)
                )

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward() #backpropagation
            scaler.step(optimizer) #update weights
            scaler.update()

            total_loss += loss.item()

            if step % args.log_every == 0 or step == len(train_loader):
                print(f"\r  epoch {epoch + 1}  batch {step}/{len(train_loader)}"
                      f"  loss {total_loss / step:.4f}", end="", flush=True)

        train_loss = total_loss / len(train_loader)
        val_loss = evaluate_loss(model, test_loader, criterion, device,
                                 PAD_SRC, PAD_TGT)
        elapsed = time.time() - started

        print(f"\rEpoch {epoch + 1}, Loss: {train_loss:.4f}   "
              f"Val loss: {val_loss:.4f}   ({elapsed:.1f}s)" + " " * 20)

        save_checkpoint(args.out, model, vocab_src, vocab_tgt, config,
                        optimizer=optimizer, epoch=epoch + 1, loss=train_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            save_checkpoint(best_path, model, vocab_src, vocab_tgt, config,
                            optimizer=optimizer, epoch=epoch + 1, loss=val_loss)

        if args.sample_sentence:
            translation = greedy_decode(model, args.sample_sentence,
                                        vocab_src, vocab_tgt, max_len, device)
            print(f"    sample: {args.sample_sentence}")
            print(f"        ->  {translation}")

    show.banner("DONE")
    show.kv("last checkpoint", args.out)
    show.kv("best checkpoint", f"{best_path}  (val loss {best_loss:.4f})")
    show.note("translate with:")
    show.note(f'  python translate.py --sentence "{args.sample_sentence}"')


if __name__ == "__main__":
    try:
        main()
    except torch.cuda.OutOfMemoryError:
        print("\n\nCUDA out of memory. Try a smaller footprint, e.g.:")
        print("  python train.py --batch-size 8 --amp")
        print("  python train.py --batch-size 8 --d-model 256 --d-ff 1024 "
              "--num-layers 3")
        sys.exit(1)


Writing train.py


### `translate.py`


In [ ]:
%%writefile translate.py


import argparse
import os

import gpu
import show
from checkpoint import load_checkpoint
from data import load_pairs
from decoding import translate as decode


def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--checkpoint", default="checkpoints/transformer_en_hi.pt")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--device", default="auto",
                   help="auto (freest GPU) / cuda:2 / cpu")
    p.add_argument("--list-gpus", action="store_true")

    p.add_argument("--sentence", default=None)
    p.add_argument("--interactive", action="store_true")
    p.add_argument("--bleu", action="store_true")
    p.add_argument("--walkthrough", action="store_true")

    p.add_argument("--method", default="greedy",
                   choices=["greedy", "sample", "beam"])
    p.add_argument("--max-len", type=int, default=50)
    p.add_argument("--temperature", type=float, default=1.0)
    p.add_argument("--top-k", type=int, default=None)
    p.add_argument("--top-p", type=float, default=None)
    p.add_argument("--beam-size", type=int, default=5)

    return p.parse_args()


def run_bleu(model, args, vocab_src, vocab_tgt, max_len, device):
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

    smooth = SmoothingFunction().method1

    test_en, test_hi = load_pairs(os.path.join(args.data_dir, "test.jsonl"))
    bleu_scores = []

    for i, (src_text, tgt_text) in enumerate(zip(test_en, test_hi)):
        pred_text = decode(
            model, src_text, vocab_src, vocab_tgt, max_len, device,
            method=args.method, temperature=args.temperature,
            top_k=args.top_k, top_p=args.top_p, beam_size=args.beam_size,
        )

        reference = [tgt_text.split()]
        hypothesis = pred_text.split()

        bleu = sentence_bleu(reference, hypothesis, smoothing_function=smooth)
        bleu_scores.append(bleu)

        print("=" * 60)
        print(f"Sample {i + 1}")
        print(f"EN (src):  {src_text}")
        print(f"HI (gt):   {tgt_text}")
        print(f"HI (pred): {pred_text}")
        print(f"BLEU: {bleu:.4f}")

    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    print("\n" + "#" * 60)
    print(f"AVERAGE BLEU over {len(bleu_scores)} samples: {avg_bleu:.4f}")
    print("#" * 60)

    return avg_bleu


def main():
    args = parse_args()
    show.init_console()

    if args.list_gpus:
        gpu.describe_devices()
        return

    device = gpu.resolve_device(args.device)

    if not os.path.exists(args.checkpoint):
        print(f"{args.checkpoint} not found. Train first:  python train.py")
        return

    model, vocab_src, vocab_tgt, config = load_checkpoint(args.checkpoint, device)
    max_len = min(args.max_len, config["max_len"])

    show.banner("MODEL")
    show.kv("checkpoint", args.checkpoint)
    show.kv("device", device)
    show.kv("vocab (en / hi)", f"{len(vocab_src)} / {len(vocab_tgt)}")
    show.kv("decoding", args.method)

    if args.walkthrough:
        import torch.nn as nn

        from walkthrough import run_walkthrough
        train_en, train_hi = load_pairs(os.path.join(args.data_dir, "train.jsonl"))
        criterion = nn.CrossEntropyLoss(ignore_index=vocab_tgt["<pad>"])
        run_walkthrough(model, train_en, train_hi, vocab_src, vocab_tgt,
                        config["max_len"], device, criterion)
        return

    if args.bleu:
        run_bleu(model, args, vocab_src, vocab_tgt, max_len, device)
        return

    def show_translation(sentence):
        translation = decode(
            model, sentence, vocab_src, vocab_tgt, max_len, device,
            method=args.method, temperature=args.temperature,
            top_k=args.top_k, top_p=args.top_p, beam_size=args.beam_size,
        )
        print(f"English: {sentence}")
        print(f"Hindi:   {translation}\n")

    if args.interactive:
        show.banner("INTERACTIVE - blank line or Ctrl+C to quit")
        while True:
            try:
                sentence = input("EN> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if not sentence:
                break
            show_translation(sentence)
        return

    show.banner("TRANSLATION")
    show_translation(args.sentence or "an empty slot on the tableau")


if __name__ == "__main__":
    main()


Writing translate.py


### `gpu.py`



In [ ]:
%%writefile gpu.py


import torch

import show


def _free_total(index):
    """Return (free_bytes, total_bytes) for a CUDA device, or None if it errors."""
    try:
        return torch.cuda.mem_get_info(index)
    except Exception:
        return None


def list_devices():
    """Rows of (index, name, free_gb, total_gb, ok) for every visible GPU."""
    rows = []

    if not torch.cuda.is_available():
        return rows

    for i in range(torch.cuda.device_count()):
        try:
            name = torch.cuda.get_device_properties(i).name
        except Exception:
            rows.append((i, "<unreadable>", 0.0, 0.0, False))
            continue

        info = _free_total(i)
        if info is None:
            rows.append((i, name, 0.0, 0.0, False))
        else:
            free, total = info
            rows.append((i, name, free / 1024**3, total / 1024**3, True))

    return rows


def describe_devices():
    show.banner("VISIBLE GPUS")
    show.kv("torch", torch.__version__)
    show.kv("cuda build", torch.version.cuda)

    rows = list_devices()

    if not rows:
        show.note("no CUDA devices visible")
        return

    print()
    print(f"    {'idx':<5}{'name':<28}{'free':>10}{'total':>10}   status")
    for i, name, free, total, ok in rows:
        status = "ok" if ok else "ERROR - skipped"
        print(f"    {i:<5}{name:<28}{free:>9.1f}G{total:>9.1f}G   {status}")


def resolve_device(spec=None, min_free_gb=2.0):
    """
    spec: None / "auto" -> the healthy CUDA device with the most free memory
          "cpu"         -> CPU
          "cuda:2" etc. -> that device
    """
    if spec == "cpu":
        return torch.device("cpu")

    if spec and spec not in ("auto", "cuda"):
        return torch.device(spec)

    if not torch.cuda.is_available():
        return torch.device("cpu")

    usable = [r for r in list_devices() if r[4] and r[2] >= min_free_gb]

    if not usable:
        show.note(f"no CUDA device has {min_free_gb:.0f} GB free - using CPU")
        return torch.device("cpu")

    best = max(usable, key=lambda r: r[2])
    return torch.device(f"cuda:{best[0]}")


def describe_selected(device):
    show.banner("DEVICE")
    show.kv("torch", torch.__version__)
    show.kv("device", device)

    if device.type == "cuda":
        index = device.index if device.index is not None else torch.cuda.current_device()
        props = torch.cuda.get_device_properties(index)
        info = _free_total(index)

        show.kv("gpu", f"cuda:{index}  {props.name}")
        show.kv("vram", f"{props.total_memory / 1024**3:.1f} GB total"
                        + (f", {info[0] / 1024**3:.1f} GB free" if info else ""))
        show.kv("compute capability", f"{props.major}.{props.minor}")
        show.kv("cuda build", torch.version.cuda)
    else:
        show.note("CUDA is not available - this will run on the CPU and be "
                  "much slower.")
        show.note("On Linux the default wheel already includes CUDA:")
        show.note("  pip install torch")


Writing gpu.py


#Download the dataset

---



Streams the first 10100 pairs of `cfilt/iitb-english-hindi` and caches them as
`data/train.jsonl` (10000) and `data/test.jsonl` (100) - the same split the
original notebook used. Everything after this point works offline.



In [ ]:
!python -u prepare_data.py


DOWNLOADING 10100 PAIRS FROM cfilt/iitb-english-hindi
README.md: 100% 3.14k/3.14k [00:00<00:00, 10.1MB/s]
dataset_infos.json: 100% 953/953 [00:00<00:00, 5.03MB/s]
    train pairs                  10000  ->  data/train.jsonl
    test pairs                   100  ->  data/test.jsonl

------------------------------------------------------------------------------
ONE EXAMPLE FROM THE DOWNLOADED DATA
------------------------------------------------------------------------------
    EN                           Highlight duration
    HI                           अवधि को हाइलाइट रकें
'[Errno 9] Bad file descriptor' thrown while requesting GET https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/321516f50bdcc1214fa75164c545478976ed84bd/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


## Training


Checkpoints are written every epoch:

- `checkpoints/transformer_en_hi.pt` - the latest epoch
- `checkpoints/transformer_en_hi_best.pt` - the best validation loss



In [ ]:
!python -u train.py --epochs 30 --batch-size 32 --amp --no-walkthrough

If Colab disconnects mid-run, pick up where it stopped:

```python
!python -u train.py --resume checkpoints/transformer_en_hi.pt --epochs 30 --no-walkthrough
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


!mkdir -p checkpoints
!cp "/content/drive/MyDrive/transformer_en_hi/transformer_en_hi.pt" checkpoints/
!ls -lh checkpoints

In [ ]:
import torch

ckpt = torch.load("checkpoints/transformer_en_hi.pt",
                  map_location="cpu", weights_only=False)
ckpt.pop("optimizer_state_dict", None)
torch.save(ckpt, "checkpoints/transformer_en_hi.pt")

!ls -lh checkpoints

**Walkthrough, with trained weights**


In [ ]:
!python -u translate.py --walkthrough

## 7. Translate

Inference loads the checkpoint and never touches the training code.

In [ ]:
!python -u translate.py --sentence "an empty slot on the tableau"

Beam search keeps the top-K hypotheses by log-probability:

In [ ]:
!python -u translate.py --method beam --beam-size 5 --sentence "an empty slot on the tableau"

Sampling, with temperature and nucleus (top-p) filtering:

In [ ]:
!python -u translate.py --method sample --temperature 1.5 --top-p 0.9 --sentence "an empty slot on the tableau"

## 8. BLEU on the test split

Decodes all 100 held-out pairs and prints per-sentence BLEU plus the average.

In [ ]:
!python -u translate.py --bleu

## 9. Translate interactively

Loading the checkpoint into the notebook itself keeps the model in memory, so
repeated translations are instant.

In [ ]:
import gpu
import show
from checkpoint import load_checkpoint
from decoding import translate as decode

show.init_console()

device = gpu.resolve_device("auto")
model, vocab_src, vocab_tgt, config = load_checkpoint(
    "checkpoints/transformer_en_hi.pt", device
)

MAX_LEN = min(50, config["max_len"])


def translate_sentence(sentence, method="greedy", **kwargs):
    hindi = decode(
        model, sentence, vocab_src, vocab_tgt, MAX_LEN, device,
        method=method, **kwargs
    )
    print(f"EN: {sentence}")
    print(f"HI: {hindi}")
    return hindi


translate_sentence("an empty slot on the tableau")

In [ ]:
#@title Translate { display-mode: "form" }
sentence = "an empty slot on the tableau"  #@param {type:"string"}
method = "greedy"  #@param ["greedy", "beam", "sample"]

translate_sentence(sentence, method=method)




**Download to your machine:**

In [ ]:
from google.colab import files

files.download("checkpoints/transformer_en_hi.pt")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!cp -r "/content/drive/MyDrive/transformer_en_hi/checkpoints" .
!cp -r "/content/drive/MyDrive/transformer_en_hi/data" .

!python -u translate.py --sentence "an empty slot on the tableau"